# Day06 예제 · server.py + client.py 를 한 노트북에

이 노트북은 **하나지만**, 실제 VS Code 프로젝트에선 **`server.py`(도구)** 와 **`client.py`(에이전트)** 두 파일로 나뉜다. 아래 셀의 `# ===== server.py =====` / `# ===== client.py =====` 주석이 그 경계다.

순서: **① 도구 직접 호출 → ② LLM 기본 호출 → ③ LLM이 도구를 스스로 호출(에이전트)**. 노트북은 두 코드가 같은 프로세스라 `Client(mcp)` 로 바로 연결된다.

## 설치

In [ ]:
%pip install -q fastmcp openai

## server.py — 도구 정의 (MCP 서버)
읽기 도구 `search_docs`, 부작용 도구 `add_task`, 친절한 에러가 있는 `set_priority`.

In [ ]:
# ===== server.py =====
# 이 셀의 내용이 그대로 server.py 가 된다. (도구를 정의하는 MCP 서버)
from fastmcp import FastMCP, Client

mcp = FastMCP("MyServer")

DOCS = [
    {"id": "vacation", "text": "연차는 사용 3영업일 전에 사내포털에서 신청한다."},
    {"id": "remote",   "text": "재택근무는 주 2회까지 가능하며 전날 18시까지 공유한다."},
]
TASKS = []
LEVELS = {"낮음", "보통", "높음"}

@mcp.tool
def search_docs(query: str, k: int = 2) -> list:
    """질문과 관련된 사내 문서를 찾는다 (읽기)"""
    scored = sorted(DOCS, key=lambda d: -sum(w in d["text"] for w in query.split()))
    return scored[:k]

@mcp.tool
def add_task(title: str) -> dict:
    """할 일을 추가한다 (부작용)"""
    TASKS.append({"id": len(TASKS) + 1, "title": title, "level": "보통"})
    return TASKS[-1]

@mcp.tool
def set_priority(task_id: int, level: str) -> dict:
    """작업 우선순위를 바꾼다. level은 낮음·보통·높음 중 하나 (아니면 친절한 에러)."""
    if level not in LEVELS:
        raise ValueError(f"level은 {sorted(LEVELS)} 중 하나여야 합니다. 받은 값: '{level}'")
    for t in TASKS:
        if t["id"] == task_id:
            t["level"] = level
            return t
    raise ValueError(f"{task_id}번 작업이 없습니다.")

# 파일로 쓸 땐 맨 아래에:
#   if __name__ == "__main__":
#       mcp.run()          # stdio.  HTTP: mcp.run(transport="http", port=8000)
print("server 준비됨 — 도구:", ["search_docs", "add_task", "set_priority"])

## ① 도구를 직접 호출 (LLM 없이)
서버가 잘 도는지 먼저 확인. `Client(mcp)` 로 붙어 `call_tool` 한다.

In [ ]:
# ① 도구를 'LLM 없이' 직접 호출 — 서버가 잘 도는지 먼저 확인 (노트북은 in-process라 Client(mcp))
async with Client(mcp) as c:
    print("도구 목록:", [t.name for t in await c.list_tools()])
    r = await c.call_tool("search_docs", {"query": "연차 신청", "k": 1})
    print("search_docs 결과:", r.data)

## ② LLM 호출 (가장 기본)
NVIDIA API로 Qwen을 부르는 최소 형태 — OpenAI 호환이라 `openai` 라이브러리를 그대로 쓴다.

In [ ]:
# ===== client.py (앞부분) — LLM 준비 =====
# NVIDIA API로 Qwen(LLM)을 부른다. OpenAI 호환이라 openai 라이브러리를 그대로 쓴다.
import os, getpass, json
from openai import OpenAI

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...): ")
llm = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)  # ← base_url만 NVIDIA
LLM_MODEL = "qwen/qwen3-next-80b-a3b-instruct"     # non-thinking 모델
print("LLM 준비됨:", LLM_MODEL)

In [ ]:
# ② LLM 호출의 '가장 기본' — 도구 없이 질문 하나만 (이게 LLM 부르는 최소 형태)
r = llm.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "MCP(Model Context Protocol)를 한 문장으로 설명해줘."}],
    temperature=0.3,
    max_tokens=200,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},   # thinking 끄기(토큰 폭주 방지)
)
print(r.choices[0].message.content)

## ③ client.py — LLM이 도구를 스스로 호출 (에이전트)
②의 LLM 호출에 `tools=` 를 넘기면, Qwen이 어떤 도구를 부를지 골라 준다. 그 도구를 실행해 결과를 되먹이는 걸 반복 = 에이전트.

In [ ]:
# ===== client.py (본체) — LLM이 도구를 '스스로' 호출하는 에이전트 =====
def to_openai_tools(tools):
    # MCP 도구(name·description·inputSchema) → OpenAI function 스펙으로 변환
    return [{"type": "function", "function": {
        "name": t.name, "description": t.description or "", "parameters": t.inputSchema}} for t in tools]

async def run_agent(question, max_steps=5):
    async with Client(mcp) as c:                 # 개발: in-process(Client(mcp)) / 배포: Client("https://<이름>.fastmcp.app/mcp")
        tools = to_openai_tools(await c.list_tools())
        messages = [{"role": "system", "content": "필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            m = llm.chat.completions.create(      # ②와 같은 호출인데 tools= 를 넘긴다
                model=LLM_MODEL, messages=messages, tools=tools, temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:                 # 도구를 더 안 부르면 최종 답변
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:              # LLM이 고른 도구를 실제 실행
                args = json.loads(tc.function.arguments)
                print(f"  [LLM → 도구] {tc.function.name}({args})")
                try:
                    out = (await c.call_tool(tc.function.name, args)).data
                except Exception as e:           # 에러도 되먹여 스스로 고치게
                    out = f"오류: {str(e).splitlines()[-1]}"
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"

# 파일로 쓸 땐 맨 아래에:
#   if __name__ == "__main__":
#       import asyncio; print(asyncio.run(run_agent("연차는 며칠 전에 신청해?")))

In [ ]:
# ③ 실행 — 노트북에선 asyncio.run 없이 그냥 await
print("Q: 연차는 며칠 전에 신청해?")
print("A:", await run_agent("연차는 며칠 전에 신청해?"))

# 멀티스텝 + 자기수정: '긴급'은 허용값이 아니라 친절한 에러 → LLM이 '높음'으로 고쳐 다시 부른다
print("\nQ: '발표 준비' 할 일을 추가하고, 그걸 '긴급'으로 설정해줘")
print("A:", await run_agent("'발표 준비' 할 일을 추가하고, 그걸 '긴급'으로 설정해줘"))

## 실제 파일로 나눌 때 — 실행 가이드

위에서 `# ===== server.py =====` / `# ===== client.py =====` 로 나눈 코드를 각각 `server.py`, `client.py` 파일로 저장하면 VS Code 프로젝트가 된다.

**1) 준비 (conda)**
```bash
conda create -n my-mcp python=3.11 -y      # 환경 생성
conda activate my-mcp                       # 활성화 (VS Code 우하단에서 이 환경을 인터프리터로 선택)
pip install fastmcp openai python-dotenv
# .env 파일에:  NVIDIA_API_KEY=nvapi-...   (반드시 .gitignore)
```

**2) `server.py` 맨 아래에 추가**
```python
if __name__ == "__main__":
    mcp.run()          # stdio.  HTTP: mcp.run(transport="http", port=8000)
```
그리고 `client.py` 위쪽 주석 `# from server import mcp` 를 실제 import 로 살린다.

**3) 도구 확인 · 실행 (in-process)**
```bash
fastmcp inspect server.py     # 도구가 잘 뜨는지 확인
python client.py              # client.py 가 server.py 의 mcp 를 import 해서 실행
```

**4) 배포처럼 HTTP로 붙일 때**
```bash
# 터미널 1 — 서버
fastmcp run server.py --transport http --port 8000
```
```python
# client.py 에서 Client 인자만 교체
async with Client("http://localhost:8000/mcp") as c:   # Client(mcp) 대신
    ...
```
```bash
# 터미널 2 — 클라이언트
python client.py
```

**5) Horizon 배포 후**
```python
async with Client("https://<이름>.fastmcp.app/mcp") as c:
    ...
```

> 핵심: **server/client 로직은 그대로, `Client(...)` 인자만 바꾼다.** (노트북=in-process → 로컬 HTTP → 배포 URL)